# Mars Transfer Optimization with Porkchop Plots

This notebook demonstrates how to find optimal launch windows for Earth-Mars transfers using porkchop plots.

## What is a Porkchop Plot?
A porkchop plot shows the delta-V requirements for an interplanetary transfer as a function of:
- Departure date (x-axis)
- Arrival date or time-of-flight (y-axis)

The contours reveal optimal launch windows where delta-V is minimized.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../space_state')

from base import BODIES, get_body_position, get_body_velocity
from guidance.lambert import lambert_solve
from guidance.porkchop import generate_porkchop, PorkchopResult

## 1. Understand Earth-Mars Synodic Period

The synodic period determines how often optimal launch windows occur.

In [ ]:
# Orbital periods
T_earth_years = 1.0
T_mars_years = 1.881

# Synodic period: 1/T_syn = |1/T_earth - 1/T_mars|
T_synodic_years = 1 / abs(1/T_earth_years - 1/T_mars_years)

print(f"Earth orbital period: {T_earth_years:.3f} years")
print(f"Mars orbital period: {T_mars_years:.3f} years")
print(f"Synodic period: {T_synodic_years:.2f} years ({T_synodic_years * 365.25:.0f} days)")
print(f"\nOptimal Mars launch windows occur approximately every {T_synodic_years:.1f} years")

## 2. Generate Porkchop Plot

We'll generate a porkchop plot spanning ~2 years to capture a complete launch window.

In [ ]:
# Define search range
# Note: Times are in days from simulation epoch
departure_start = 0     # Start searching from t=0
departure_end = 365     # Search over 1 year of departures

# Time of flight range
tof_min = 150   # Minimum 150 days
tof_max = 400   # Maximum 400 days

# Generate porkchop data
print("Generating porkchop plot data...")
print(f"Departure range: {departure_start} to {departure_end} days")
print(f"TOF range: {tof_min} to {tof_max} days")

In [ ]:
# Generate the porkchop data
porkchop = generate_porkchop(
    departure_body='earth',
    arrival_body='mars barycenter',
    departure_start=departure_start,
    departure_end=departure_end,
    tof_min=tof_min,
    tof_max=tof_max,
    n_departure=50,
    n_tof=50,
)

print(f"\nPorkchop data generated")
print(f"Grid size: {porkchop.departure_dates.shape[0]} x {porkchop.tof.shape[0]}")

## 3. Visualize Porkchop Plot

In [ ]:
# Convert delta-V from AU/day to km/s
dv_kmps = porkchop.total_dv * 1731.46

# Create figure
fig, ax = plt.subplots(figsize=(12, 8))

# Create meshgrid for plotting
D, T = np.meshgrid(porkchop.departure_dates, porkchop.tof)

# Clip very high values for better visualization
dv_plot = np.clip(dv_kmps.T, 0, 20)

# Contour plot
levels = np.arange(5, 15, 0.5)  # 5 to 15 km/s in 0.5 km/s steps
cs = ax.contour(D, T, dv_plot, levels=levels, colors='black', linewidths=0.5)
ax.clabel(cs, inline=True, fontsize=8, fmt='%.1f')

# Filled contour
cf = ax.contourf(D, T, dv_plot, levels=np.linspace(5, 15, 50), cmap='RdYlGn_r', extend='both')
plt.colorbar(cf, ax=ax, label='Total Delta-V (km/s)')

# Find and mark minimum
min_idx = np.unravel_index(np.argmin(dv_kmps), dv_kmps.shape)
opt_dep = porkchop.departure_dates[min_idx[0]]
opt_tof = porkchop.tof[min_idx[1]]
opt_dv = dv_kmps[min_idx]

ax.scatter([opt_dep], [opt_tof], c='white', s=100, marker='*', edgecolors='black', linewidths=2, zorder=5)
ax.annotate(f'Optimal\n{opt_dv:.1f} km/s', (opt_dep, opt_tof), 
            xytext=(10, 10), textcoords='offset points', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax.set_xlabel('Departure Date (days from epoch)', fontsize=12)
ax.set_ylabel('Time of Flight (days)', fontsize=12)
ax.set_title('Earth-Mars Transfer Porkchop Plot', fontsize=14)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nOptimal transfer found:")
print(f"  Departure: Day {opt_dep:.0f}")
print(f"  Time of flight: {opt_tof:.0f} days")
print(f"  Total delta-V: {opt_dv:.2f} km/s")

## 4. Analyze Optimal Transfer

In [ ]:
# Get detailed solution for optimal transfer
mu_sun = BODIES['sun']['mu']

# Get positions at departure and arrival
r1 = get_body_position('earth', opt_dep)
r2 = get_body_position('mars barycenter', opt_dep + opt_tof)

# Solve Lambert problem
sol = lambert_solve(r1, r2, opt_tof, mu_sun)

# Get planet velocities
v_earth = get_body_velocity('earth', opt_dep)
v_mars = get_body_velocity('mars barycenter', opt_dep + opt_tof)

# Delta-V magnitudes (km/s)
dv1_kmps = np.linalg.norm(sol.v1 - v_earth) * 1731.46
dv2_kmps = np.linalg.norm(sol.v2 - v_mars) * 1731.46

print("Optimal Transfer Details:")
print("=" * 50)
print(f"Departure delta-V: {dv1_kmps:.2f} km/s (Earth escape)")
print(f"Arrival delta-V: {dv2_kmps:.2f} km/s (Mars capture)")
print(f"Total delta-V: {dv1_kmps + dv2_kmps:.2f} km/s")
print(f"")
print(f"Transfer orbit:")
print(f"  Semi-major axis: {sol.a:.3f} AU")
print(f"  Eccentricity: {sol.e:.3f}")
print(f"  Type: {'Prograde' if sol.prograde else 'Retrograde'}")

## 5. Plot Transfer Trajectory

In [ ]:
# Generate transfer trajectory points
n_points = 100
times = np.linspace(opt_dep, opt_dep + opt_tof, n_points)

# For a proper trajectory, we'd integrate, but we can approximate with the Lambert solution
# by interpolating the conic arc

# Get Earth and Mars orbits for reference
earth_orbit = np.array([get_body_position('earth', t) for t in np.linspace(0, 365, 100)])
mars_orbit = np.array([get_body_position('mars barycenter', t) for t in np.linspace(0, 687, 100)])

# Get departure and arrival positions
earth_dep = get_body_position('earth', opt_dep)
mars_arr = get_body_position('mars barycenter', opt_dep + opt_tof)

# Plot
fig, ax = plt.subplots(figsize=(10, 10))

# Sun at center
ax.scatter([0], [0], c='yellow', s=500, edgecolors='orange', linewidths=2, zorder=10, label='Sun')

# Planet orbits
ax.plot(earth_orbit[:, 0], earth_orbit[:, 1], 'b-', alpha=0.5, label='Earth orbit')
ax.plot(mars_orbit[:, 0], mars_orbit[:, 1], 'r-', alpha=0.5, label='Mars orbit')

# Departure and arrival points
ax.scatter(*earth_dep[:2], c='blue', s=100, zorder=5, label=f'Earth (departure)')
ax.scatter(*mars_arr[:2], c='red', s=100, zorder=5, label=f'Mars (arrival)')

# Transfer arc (simple straight line approximation - actual arc would be curved)
ax.plot([earth_dep[0], mars_arr[0]], [earth_dep[1], mars_arr[1]], 'g--', linewidth=2, 
        label='Transfer trajectory')

# Velocity vectors (scaled)
scale = 50  # Scale factor for velocity arrows
ax.arrow(earth_dep[0], earth_dep[1], sol.v1[0]*scale, sol.v1[1]*scale, 
         head_width=0.05, head_length=0.02, fc='green', ec='green')

ax.set_xlabel('X (AU)', fontsize=12)
ax.set_ylabel('Y (AU)', fontsize=12)
ax.set_title('Earth-Mars Transfer Trajectory', fontsize=14)
ax.legend(loc='upper left')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_xlim(-2, 2)
ax.set_ylim(-2, 2)

plt.tight_layout()
plt.show()

## 6. Compare Type I vs Type II Transfers

Type I transfers have TOF < 180 days (short way)
Type II transfers have TOF > 180 days (long way)

In [ ]:
# Find best Type I and Type II transfers
type1_mask = porkchop.tof < 270  # Roughly Type I
type2_mask = porkchop.tof >= 270  # Roughly Type II

# Best Type I
type1_dv = np.where(type1_mask[:, np.newaxis], dv_kmps.T, np.inf)
type1_min_idx = np.unravel_index(np.argmin(type1_dv), type1_dv.shape)

# Best Type II
type2_dv = np.where(type2_mask[:, np.newaxis], dv_kmps.T, np.inf)
type2_min_idx = np.unravel_index(np.argmin(type2_dv), type2_dv.shape)

print("Transfer Type Comparison:")
print("=" * 50)
print(f"Type I (shorter arc):")
print(f"  TOF: {porkchop.tof[type1_min_idx[0]]:.0f} days")
print(f"  Delta-V: {type1_dv[type1_min_idx]:.2f} km/s")
print(f"")
print(f"Type II (longer arc):")
print(f"  TOF: {porkchop.tof[type2_min_idx[0]]:.0f} days")
print(f"  Delta-V: {type2_dv[type2_min_idx]:.2f} km/s")

## 7. Mission Fuel Requirements

In [ ]:
# Calculate fuel requirements using Tsiolkovsky equation
# Δv = Isp * g0 * ln(m0/mf)
# m_propellant = m0 * (1 - exp(-Δv / (Isp * g0)))

g0 = 9.80665  # m/s²

# Example: Send 10,000 kg payload to Mars
payload_mass = 10000  # kg

# Different engine options
engines = {
    'Chemical (Isp=450s)': 450,
    'Nuclear Thermal (Isp=900s)': 900,
    'Ion Drive (Isp=3000s)': 3000,
}

total_dv = opt_dv * 1000  # Convert to m/s

print(f"Fuel Requirements for {payload_mass:,} kg payload to Mars")
print(f"Total delta-V: {total_dv:.0f} m/s")
print("=" * 50)

for name, isp in engines.items():
    # Mass ratio
    mass_ratio = np.exp(total_dv / (isp * g0))
    
    # Initial mass required
    m0 = payload_mass * mass_ratio
    
    # Propellant mass
    m_prop = m0 - payload_mass
    
    print(f"\n{name}:")
    print(f"  Mass ratio: {mass_ratio:.2f}")
    print(f"  Propellant required: {m_prop:,.0f} kg")
    print(f"  Initial mass: {m0:,.0f} kg")

## Summary

Key findings from this analysis:
1. Optimal Earth-Mars launch windows occur approximately every 26 months
2. Minimum delta-V for Mars transfer is typically 5-6 km/s total
3. Type I transfers are faster but may require more delta-V
4. Chemical propulsion requires significant fuel mass for Mars missions